# Notebook 6 -- Generative AI with Transformers

## From Classifying to Creating: The Generative AI Revolution

Welcome to the **capstone notebook** of this deep-learning series. Over the previous
five notebooks you built autograd engines, trained MLPs and CNNs, studied training
dynamics, and built tiny generative models from scratch in pure NumPy.

Now we bring it all together. In this notebook you will:

1. **Implement self-attention from scratch** in PyTorch -- the mechanism that powers
   every modern generative model.
2. **Build a minimal GPT-like transformer** and train it on Shakespeare text so you
   can watch it learn to write (sort of).
3. **Use a pre-trained language model** (DistilGPT-2) via HuggingFace and explore
   how temperature, top-k, and top-p control generation.
4. **Fine-tune** DistilGPT-2 on a small domain-specific corpus.
5. **Build a Gradio demo** -- a web UI you can share.
6. **Reflect** on what deep learning is and isn't, tying back every curriculum point
   from the entire series.

> **Prerequisites**: You should have completed Notebooks 1-5. We will reference them
> frequently with "Remember when you built X from scratch? Here is how it scales."

| Section | What We Build | Key Concept |
|---------|--------------|-------------|
| 1 | Scaled dot-product & multi-head attention | Dynamic input weighting |
| 2 | Tiny GPT (2 layers, 2 heads, 64 dim) | Transformer architecture |
| 3 | HuggingFace text-generation pipeline | Pretrained models |
| 4 | Fine-tuned DistilGPT-2 | Transfer learning |
| 5 | Gradio web demo | Deployment |
| 6 | Reflection | All 7 curriculum points |


---
## Setup & Imports

In [ ]:
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cpu")  # All code runs on CPU
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")


---
# Part 1 -- Self-Attention from Scratch

## The Core Idea

In a standard MLP or CNN, every input position interacts with the same fixed set of
weights. **Attention** is different: it lets each position *dynamically decide* which
other positions to look at, based on the content of those positions.

### The Math

Given a sequence of embeddings $X \in \mathbb{R}^{T \times d}$, we compute three
matrices via learned projections:

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Then: $\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$

The $\frac{1}{\sqrt{d_k}}$ scaling prevents the dot products from becoming too large
(which would push softmax into saturation -- recall the numerical stability issues from
Notebook 3).


In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention.

    Args:
        Q: queries  (batch, seq_len, d_k)
        K: keys     (batch, seq_len, d_k)
        V: values   (batch, seq_len, d_v)
        mask: optional (seq_len, seq_len) -- 0 means "don't attend"

    Returns:
        output: (batch, seq_len, d_v)
        weights: (batch, seq_len, seq_len) attention weights
    """
    d_k = Q.size(-1)

    # Step 1: Compute raw attention scores
    # Q @ K^T gives (batch, seq_len, seq_len) -- how much each position
    # "cares about" each other position
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    # Step 2: Apply mask (if provided) -- set masked positions to -inf
    # so softmax assigns them probability ~0
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    # Step 3: Softmax over the last dim -- each row sums to 1
    weights = F.softmax(scores, dim=-1)

    # Step 4: Weighted sum of values
    output = torch.matmul(weights, V)

    return output, weights


# --- Demo: attention on a tiny sequence ---
torch.manual_seed(SEED)
seq_len, d_model = 6, 8
X = torch.randn(1, seq_len, d_model)  # batch=1, 6 tokens, dim=8

# For this demo, Q=K=V=X (self-attention without learned projections)
output, weights = scaled_dot_product_attention(X, X, X)

print("Input shape: ", X.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", weights.shape)
print()
print("Attention weights (each row sums to 1):")
print(weights[0].detach().numpy().round(3))
print()
print("Row sums:", weights[0].sum(dim=-1).detach().numpy().round(6))


### Visualizing Attention Weights

Attention weights tell us: "For position *i*, how much does it attend to position *j*?"
Let's plot them as a heatmap.

In [ ]:
def plot_attention_weights(weights, title="Attention Weights", tokens=None):
    """Plot attention weights as a heatmap."""
    w = weights.detach().numpy()
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(w, cmap="Blues", vmin=0, vmax=w.max())
    ax.set_xlabel("Key position")
    ax.set_ylabel("Query position")
    ax.set_title(title)
    if tokens:
        ax.set_xticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=45, ha="right")
        ax.set_yticks(range(len(tokens)))
        ax.set_yticklabels(tokens)
    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


# Visualize the attention weights from our demo
tokens = ["The", "cat", "sat", "on", "the", "mat"]
plot_attention_weights(weights[0], "Self-Attention Weights", tokens)


### Causal (Autoregressive) Masking

For language generation (GPT-style), position $i$ must only attend to positions
$\leq i$ -- it cannot look into the future. We enforce this with a **causal mask**:
a lower-triangular matrix of ones.

In [ ]:
# Create causal mask: lower triangular
causal_mask = torch.tril(torch.ones(seq_len, seq_len))
print("Causal mask:")
print(causal_mask.numpy().astype(int))

# Apply it
output_causal, weights_causal = scaled_dot_product_attention(X, X, X, mask=causal_mask)
plot_attention_weights(weights_causal[0], "Causal (Masked) Attention", tokens)

print("Notice: each position can only attend to itself and earlier positions.")
print("This is exactly how GPT generates text -- one token at a time, left to right.")


## Multi-Head Attention

One attention head can only focus on one pattern at a time. **Multi-head attention**
runs several attention heads in parallel, each with its own Q/K/V projections, then
concatenates their outputs.

This lets the model simultaneously attend to different types of relationships
(e.g., one head for syntactic structure, another for semantic similarity).

$$\text{MultiHead}(X) = \text{Concat}(\text{head}_1, \dots, \text{head}_h) W_O$$


In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention -- the core building block of transformers.

    Remember from Notebook 1: you built a Neuron class that computed wx+b.
    This is the same idea, but instead of a single weighted sum, we compute
    *multiple* weighted sums in parallel (the heads), where the weights are
    determined dynamically by the input content (attention).
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # dimension per head

        # Learned projections for Q, K, V, and output
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape

        # Project input into Q, K, V
        Q = self.W_q(x)  # (batch, seq_len, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        # Reshape to (batch, n_heads, seq_len, d_k) -- split d_model into heads
        Q = Q.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        # Scaled dot-product attention per head
        d_k = self.d_k
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        weights = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, V)

        # Concatenate heads and project back
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        output = self.W_o(attn_out)

        return output, weights


# Test multi-head attention
torch.manual_seed(SEED)
mha = MultiHeadAttention(d_model=8, n_heads=2)
causal = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
out, attn_weights = mha(X, mask=causal)

print(f"Input:  {X.shape}")
print(f"Output: {out.shape}")
print(f"Attention weights: {attn_weights.shape}  (batch, heads, seq, seq)")


In [ ]:
# Visualize each attention head
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for h in range(2):
    w = attn_weights[0, h].detach().numpy()
    im = axes[h].imshow(w, cmap="Blues", vmin=0)
    axes[h].set_title(f"Head {h+1}")
    axes[h].set_xlabel("Key position")
    axes[h].set_ylabel("Query position")
    if tokens:
        axes[h].set_xticks(range(len(tokens)))
        axes[h].set_xticklabels(tokens, rotation=45, ha="right")
        axes[h].set_yticks(range(len(tokens)))
        axes[h].set_yticklabels(tokens)

plt.suptitle("Multi-Head Attention -- Each Head Learns Different Patterns", y=1.02)
plt.tight_layout()
plt.show()
print("Each head attends to different relationships in the input.")
print("This is analogous to having multiple feature detectors in a CNN (Notebook 5).")


> **KEY INSIGHT: Attention lets the model decide which parts of input to focus on**
>
> In a standard MLP (Notebook 1), every input contributes equally through fixed weights.
> In a CNN (Notebook 5), locality is hard-coded by the kernel size.
> Attention has *no such constraint* -- it can relate any position to any other position,
> and the pattern of connections is determined by the data, not the architecture.
> This is why transformers scale so well to language, images, audio, and more.


---
# Part 2 -- A Minimal Transformer (Tiny GPT)

We will now build a complete transformer language model. The architecture:

```
Input tokens
    |
    v
Token Embedding + Positional Embedding
    |
    v
[TransformerBlock x N_LAYERS]
    |     LayerNorm -> MultiHeadAttention -> Residual connection
    |     LayerNorm -> Feed-Forward Network -> Residual connection
    |
    v
LayerNorm -> Linear -> logits over vocabulary
```

Each component should look familiar from earlier notebooks:
- **Embeddings** = learned lookup table (like the character embeddings in Notebook 3)
- **LayerNorm** = normalize activations (like BatchNorm from Notebook 5, but per-sample)
- **Feed-forward** = two-layer MLP with GELU activation (like Notebook 1, but modern)
- **Residual connections** = skip connections (like ResNet intuition from Notebook 5)


In [ ]:
class TransformerBlock(nn.Module):
    """One transformer block: attention + feed-forward, each with
    layer normalization and residual connections.

    Architecture:
        x -> LayerNorm -> MultiHeadAttention -> + (residual) -> LayerNorm -> FFN -> + (residual)
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        # Feed-forward network: expand then compress
        # (d_model -> d_ff -> d_model) with GELU activation
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-norm architecture (used in modern transformers like GPT-2)
        normed = self.ln1(x)
        attn_out, _ = self.attn(normed, mask=mask)
        x = x + self.drop1(attn_out)   # Residual connection #1

        normed = self.ln2(x)
        ffn_out = self.ffn(normed)
        x = x + self.drop2(ffn_out)    # Residual connection #2

        return x


print("TransformerBlock defined.")
print("Key: LayerNorm -> Attention -> Residual -> LayerNorm -> FFN -> Residual")


## Tiny GPT: A Character-Level Transformer Language Model

Now let's assemble a complete GPT-like model. We keep it deliberately small so it
trains in a few minutes on CPU:
- **2 layers**, **2 attention heads**, embedding dimension **64**
- Block size (context window) = **64** characters
- This is roughly 100K parameters -- tiny by modern standards (GPT-3 has 175 *billion*)


In [ ]:
class TinyGPT(nn.Module):
    """A minimal GPT-style transformer for character-level language modeling.

    This is the same architecture as GPT-2, just much smaller:
    - GPT-2: 12 layers, 12 heads, 768 dim, 117M params
    - Ours:  2 layers, 2 heads, 64 dim, ~100K params
    """
    def __init__(self, vocab_size, d_model=64, n_heads=2, n_layers=2,
                 block_size=64, d_ff=256, dropout=0.1):
        super().__init__()
        self.block_size = block_size

        # Token embeddings: each character gets a learned vector
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        # Positional embeddings: each position gets a learned vector
        self.pos_emb = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)

        # Stack of transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # Final layer norm and output projection
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share embedding weights with output projection
        # (a common trick that reduces parameters and improves performance)
        self.head.weight = self.tok_emb.weight

        # Count parameters
        n_params = sum(p.numel() for p in self.parameters())
        print(f"TinyGPT: {n_params:,} parameters")
        print(f"  vocab_size={vocab_size}, d_model={d_model}, n_heads={n_heads}")
        print(f"  n_layers={n_layers}, block_size={block_size}")

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"Sequence length {T} > block_size {self.block_size}"

        # Token + positional embeddings
        tok = self.tok_emb(idx)                           # (B, T, d_model)
        pos = self.pos_emb(torch.arange(T, device=idx.device))  # (T, d_model)
        x = self.drop(tok + pos)                          # (B, T, d_model)

        # Causal mask
        mask = torch.tril(torch.ones(T, T, device=idx.device)).unsqueeze(0).unsqueeze(0)

        # Pass through transformer blocks
        for block in self.blocks:
            x = block(x, mask=mask)

        x = self.ln_f(x)
        logits = self.head(x)  # (B, T, vocab_size)

        # Compute loss if targets provided
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive generation: predict one token at a time."""
        self.eval()
        for _ in range(max_new_tokens):
            # Crop to block_size if needed
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            # Take logits at the last position
            logits = logits[:, -1, :] / temperature
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


## Shakespeare Dataset

We include a chunk of Shakespeare text directly in the notebook -- the same approach
as Notebook 3 but now we will model it with a transformer.

In [ ]:
# ~5KB of Shakespeare text embedded directly
SHAKESPEARE_TEXT = """ROMEO: But, soft! what light through yonder window breaks? It is the east, and Juliet is the sun. Arise, fair sun, and kill the envious moon, Who is already sick and pale with grief, That thou her maid art far more fair than she. Be not her maid, since she is envious; Her vestal livery is but sick and green And none but fools do wear it; cast it off. It is my lady, O, it is my love! O, that she knew she were! She speaks yet she says nothing: what of that? Her eye discourses; I will answer it. I am too bold, tis not to me she speaks. Two of the fairest stars in all the heaven, Having some business, do entreat her eyes To twinkle in their spheres till they return. JULIET: O Romeo, Romeo! wherefore art thou Romeo? Deny thy father and refuse thy name; Or, if thou wilt not, be but sworn my love, And I will no longer be a Capulet. ROMEO: Shall I hear more, or shall I speak at this? JULIET: Tis but thy name that is my enemy; Thou art thyself, though not a Montague. What is a Montague? it is nor hand, nor foot, Nor arm, nor face, nor any other part Belonging to a man. O, be some other name! What is in a name? that which we call a rose By any other name would smell as sweet; So Romeo would, were he not Romeo called, Retain that dear perfection which he owes Without that title. Romeo, doff thy name, And for that name which is no part of thee Take all myself. ROMEO: I take thee at thy word: Call me but love, and I will be new baptized; Henceforth I never will be Romeo. JULIET: What man art thou that thus bescreened in night So stumblest on my counsel? ROMEO: By a name I know not how to tell thee who I am: My name, dear saint, is hateful to myself, Because it is an enemy to thee; Had I it written, I would tear the word. JULIET: My ears have not yet drunk a hundred words Of that tongue uttering, yet I know the sound: Art thou not Romeo and a Montague? ROMEO: Neither, fair saint, if either thee dislike. HAMLET: To be, or not to be, that is the question: Whether tis nobler in the mind to suffer The slings and arrows of outrageous fortune, Or to take arms against a sea of troubles, And by opposing end them? To die: to sleep; No more; and by a sleep to say we end The heart-ache and the thousand natural shocks That flesh is heir to, tis a consummation Devoutly to be wished. To die, to sleep; To sleep: perchance to dream: ay, there is the rub; For in that sleep of death what dreams may come When we have shuffled off this mortal coil, Must give us pause: there is the respect That makes calamity of so long life; For who would bear the whips and scorns of time, The oppressor wrong, the proud man contumely, The pangs of despised love, the law delay, The insolence of office and the spurns That patient merit of the unworthy takes, When he himself might his quietus make With a bare bodkin? who would fardels bear, To grunt and sweat under a weary life, But that the dread of something after death, The undiscovered country from whose bourn No traveller returns, puzzles the will And makes us rather bear those ills we have Than fly to others that we know not of? Thus conscience does make cowards of us all; And thus the native hue of resolution Is sicklied over with the pale cast of thought, And enterprises of great pith and moment With this regard their currents turn awry, And lose the name of action. PROSPERO: Our revels now are ended. These our actors, As I foretold you, were all spirits and Are melted into air, into thin air: And, like the baseless fabric of this vision, The cloud-capped towers, the gorgeous palaces, The solemn temples, the great globe itself, Yea, all which it inherit, shall dissolve And, like this insubstantial pageant faded, Leave not a rack behind. We are such stuff As dreams are made on, and our little life Is rounded with a sleep. MACBETH: Tomorrow, and tomorrow, and tomorrow, Creeps in this petty pace from day to day To the last syllable of recorded time, And all our yesterdays have lighted fools The way to dusty death. Out, out, brief candle! Life is but a walking shadow, a poor player That struts and frets his hour upon the stage And then is heard no more: it is a tale Told by an idiot, full of sound and fury, Signifying nothing. KING LEAR: Blow, winds, and crack your cheeks! rage! blow! You cataracts and hurricanoes, spout Till you have drenched our steeples, drowned the cocks! You sulphurous and thought-executing fires, Vaunt-couriers to oak-cleaving thunderbolts, Singe my white head! And thou, all-shaking thunder, Smite flat the thick rotundity of the world! """

print(f"Corpus size: {len(SHAKESPEARE_TEXT)} characters")
print(f"Preview: {SHAKESPEARE_TEXT[:200]}...")


In [ ]:
# Build character-level vocabulary
chars = sorted(list(set(SHAKESPEARE_TEXT)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

print(f"Vocabulary size: {vocab_size}")
print("Characters:", "".join(chars))

# Encode the entire text as a tensor of integer indices
data = torch.tensor([char_to_idx[ch] for ch in SHAKESPEARE_TEXT], dtype=torch.long)
print(f"Encoded tensor shape: {data.shape}")


In [ ]:
class ShakespeareDataset(Dataset):
    """Character-level dataset: given a sequence of characters,
    predict the next character at each position.

    This is exactly the same task as Notebook 3, but now we use
    PyTorch Dataset/DataLoader instead of manual batching.
    """
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.block_size + 1]
        x = chunk[:-1]   # input: characters 0..block_size-1
        y = chunk[1:]     # target: characters 1..block_size (shifted by 1)
        return x, y


BLOCK_SIZE = 64
dataset = ShakespeareDataset(data, BLOCK_SIZE)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)
print(f"Dataset size: {len(dataset)} sequences of length {BLOCK_SIZE}")
print(f"Batches per epoch: {len(dataloader)}")


## Training the Tiny GPT

Let's train! We use a standard training loop with AdamW optimizer. The model is small
enough to train in a few minutes on CPU.

In [ ]:
# Create model
torch.manual_seed(SEED)
config = {
    "vocab_size": vocab_size,
    "d_model": 64,
    "n_heads": 2,
    "n_layers": 2,
    "block_size": BLOCK_SIZE,
    "d_ff": 256,
    "dropout": 0.1,
}
model = TinyGPT(**config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)


In [ ]:
# Training loop
NUM_EPOCHS = 3
losses = []

print("Training Tiny GPT...")
print("=" * 60)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_losses = []

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)

        _, loss = model(xb, targets=yb)

        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping (prevents exploding gradients -- Notebook 2 topic)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_losses.append(loss.item())
        losses.append(loss.item())

    avg_loss = np.mean(epoch_losses)
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f}")

    # Generate a sample to see progress
    model.eval()
    prompt = torch.tensor([[char_to_idx["R"]]], dtype=torch.long, device=device)
    gen_ids = model.generate(prompt, max_new_tokens=80, temperature=0.8)
    text = "".join(idx_to_char[i] for i in gen_ids[0].tolist())
    print(f"  Sample: {text[:80]}")
    print()

print("Training complete!")


In [ ]:
# Plot training loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(losses) + 1), losses, alpha=0.3, color="steelblue", label="Per-batch loss")
# Smoothed loss
window = max(1, len(losses) // 20)
if len(losses) > window:
    smoothed = np.convolve(losses, np.ones(window)/window, mode="valid")
    ax.plot(range(window, window + len(smoothed)), smoothed, color="darkblue", linewidth=2, label=f"Smoothed (window={window})")
ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("Tiny GPT Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Text Generation with Temperature Control

Remember from Notebook 3: **temperature** controls the sharpness of the softmax
distribution.
- **T < 1**: more conservative, repetitive, "safe" text
- **T = 1**: sample from the learned distribution as-is
- **T > 1**: more random, creative, but also more errors


In [ ]:
# Generate text at different temperatures
model.eval()
temperatures = [0.3, 0.7, 1.0, 1.5]
prompt_text = "ROMEO: "
prompt_ids = torch.tensor([[char_to_idx[c] for c in prompt_text]], dtype=torch.long, device=device)

print("=" * 70)
print("TEXT GENERATION AT DIFFERENT TEMPERATURES")
print("=" * 70)

for temp in temperatures:
    torch.manual_seed(SEED)
    gen_ids = model.generate(prompt_ids, max_new_tokens=150, temperature=temp)
    text = "".join(idx_to_char[i] for i in gen_ids[0].tolist())
    print(f"\n--- Temperature = {temp} ---")
    print(text)

print("\n" + "=" * 70)
print("Notice: lower temperature = more repetitive but safer;")
print("higher temperature = more variety but more nonsense.")
print("The model learned statistical patterns, not meaning!")


---
# Part 3 -- Using Pretrained Models with HuggingFace

Our tiny transformer learned some Shakespeare-like patterns from a few KB of text.
Real language models like GPT-2 are trained on *billions* of words from the internet.

**HuggingFace Transformers** gives us easy access to thousands of pretrained models.
Let's load **DistilGPT-2** -- a distilled (compressed) version of GPT-2 with 82M
parameters. It runs comfortably on CPU.


In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer, pipeline
import torch

# Load pre-trained DistilGPT-2
print("Loading DistilGPT-2...")
pretrained_tokenizer = GPT2Tokenizer.from_pretrained("distilgpt2")
pretrained_model = GPT2LMHeadModel.from_pretrained("distilgpt2")
pretrained_model.eval()

# Set pad token (GPT-2 doesn't have one by default)
pretrained_tokenizer.pad_token = pretrained_tokenizer.eos_token

n_params = sum(p.numel() for p in pretrained_model.parameters())
print(f"DistilGPT-2 loaded: {n_params:,} parameters")
print(f"Vocabulary size: {pretrained_tokenizer.vocab_size}")
print(f"\nCompare: our Tiny GPT has ~100K params trained on ~5KB of Shakespeare.")
print(f"DistilGPT-2 has {n_params:,} params trained on ~8GB of internet text.")


In [ ]:
# Simple text generation using the pipeline API
generator = pipeline(
    "text-generation",
    model=pretrained_model,
    tokenizer=pretrained_tokenizer,
    device=-1,  # CPU
)

prompts = [
    "The meaning of life is",
    "In the beginning, there was",
    "Once upon a time in a land far away",
]

print("=" * 70)
print("DISTILGPT-2 TEXT GENERATION")
print("=" * 70)

for prompt in prompts:
    result = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.8,
        num_return_sequences=1,
        pad_token_id=pretrained_tokenizer.eos_token_id,
    )
    print(f"\nPrompt: {prompt!r}")
    print(f"Output: {result[0]['generated_text']}")
    print("-" * 70)


### Top-k and Top-p (Nucleus) Sampling

Beyond temperature, modern LLMs use **top-k** and **top-p** sampling to control
the quality/diversity tradeoff:

- **Top-k**: only sample from the k most probable tokens
- **Top-p** (nucleus): only sample from the smallest set of tokens whose
  cumulative probability exceeds p

These prevent the model from picking extremely unlikely tokens (which cause
incoherent text) while still allowing some diversity.


In [ ]:
prompt = "Once upon a time in a land far away"

print("=" * 70)
print("TOP-K AND TOP-P SAMPLING COMPARISON")
print("=" * 70)

configs = [
    {"label": "Greedy (no sampling)", "do_sample": False},
    {"label": "Temperature=1.0 (raw)", "do_sample": True, "temperature": 1.0},
    {"label": "Top-k=50", "do_sample": True, "top_k": 50},
    {"label": "Top-p=0.9", "do_sample": True, "top_p": 0.9},
    {"label": "Top-k=50 + Top-p=0.9", "do_sample": True, "top_k": 50, "top_p": 0.9},
]

for cfg in configs:
    label = cfg.pop("label")
    result = generator(
        prompt,
        max_new_tokens=50,
        num_return_sequences=1,
        pad_token_id=pretrained_tokenizer.eos_token_id,
        **cfg,
    )
    print(f"\n[{label}]")
    print(result[0]["generated_text"])
    print("-" * 70)


> ### Curriculum Point 4: Neural Networks Don't "Understand"
>
> Look at the generated text above. It is often grammatically plausible and
> locally coherent. But does the model *understand* what it is writing?
>
> **No.** The model has learned statistical regularities: which tokens tend to follow
> which other tokens. It has no concept of truth, meaning, or the real world.
>
> Let's demonstrate this with adversarial prompts -- questions where the "pattern-
> matching" nature of the model becomes obvious.


In [ ]:
# Demonstration: the model doesn't "understand"
adversarial_prompts = [
    "2 + 2 = ",
    "The capital of the fictional country Zorgblatt is",
    "If you reverse the word hello you get",
    "The number after 999999 is",
]

print("=" * 70)
print("ADVERSARIAL PROMPTS: Testing 'understanding'")
print("=" * 70)

for prompt in adversarial_prompts:
    result = generator(
        prompt,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
        pad_token_id=pretrained_tokenizer.eos_token_id,
    )
    print(f"\nPrompt: {prompt!r}")
    print(f"Output: {result[0]['generated_text']}")
    print("-" * 70)

print()
print("KEY TAKEAWAY: The model generates plausible-*sounding* text, but:")
print("  - It may get math wrong (it is not computing, just pattern matching)")
print("  - It happily invents facts about nonexistent things")
print("  - String reversal requires algorithmic reasoning it does not have")
print("  - These failures reveal: it learned P(next_token | context), not meaning")


---
# Part 4 -- Fine-Tuning DistilGPT-2

**Fine-tuning** takes a pre-trained model and continues training on a small
domain-specific dataset. The model already knows English -- we just shift its
distribution toward our target domain.

We will fine-tune DistilGPT-2 on a small corpus of cooking recipes. This
demonstrates the power of transfer learning: instead of training from scratch
(which requires massive data and compute), we start from a model that already
"knows" language and nudge it toward our domain.


In [ ]:
# Small cooking recipe corpus for fine-tuning
RECIPE_CORPUS = """Recipe: Classic Chocolate Chip Cookies
Ingredients: 2 cups flour, 1 cup butter, 1 cup sugar, 2 eggs, 1 tsp vanilla, 1 cup chocolate chips
Instructions: Preheat oven to 375F. Cream butter and sugar. Beat in eggs and vanilla. Mix in flour. Fold in chocolate chips. Drop spoonfuls onto baking sheet. Bake 10 minutes until golden.

Recipe: Simple Tomato Pasta
Ingredients: 1 lb spaghetti, 4 tomatoes, 3 cloves garlic, olive oil, basil, salt, pepper
Instructions: Cook pasta until al dente. Dice tomatoes and mince garlic. Saute garlic in olive oil. Add tomatoes and cook 10 minutes. Season with salt and pepper. Toss with pasta. Top with fresh basil.

Recipe: Banana Bread
Ingredients: 3 ripe bananas, 1/3 cup melted butter, 3/4 cup sugar, 1 egg, 1 tsp vanilla, 1 tsp baking soda, 1.5 cups flour
Instructions: Preheat oven to 350F. Mash bananas and mix with melted butter. Stir in sugar, egg, vanilla. Add baking soda and flour. Pour into loaf pan. Bake 60 minutes.

Recipe: Greek Salad
Ingredients: 1 cucumber, 4 tomatoes, 1 red onion, 1 cup olives, feta cheese, olive oil, oregano
Instructions: Chop cucumber, tomatoes, and onion into chunks. Add olives and crumbled feta. Drizzle with olive oil. Sprinkle oregano. Toss gently. Serve immediately.

Recipe: Chicken Stir Fry
Ingredients: 2 chicken breasts, 2 cups mixed vegetables, 3 tbsp soy sauce, 1 tbsp sesame oil, garlic, ginger, rice
Instructions: Slice chicken thinly. Cook rice. Heat sesame oil in wok. Stir fry chicken until cooked. Add vegetables and garlic. Add soy sauce. Serve over rice.

Recipe: Homemade Pizza
Ingredients: pizza dough, 1 cup tomato sauce, 2 cups mozzarella, toppings of choice
Instructions: Preheat oven to 450F. Roll out dough on floured surface. Spread sauce evenly. Add mozzarella. Add toppings. Bake 12 minutes until crust is golden and cheese bubbles.

Recipe: Vegetable Soup
Ingredients: 2 carrots, 2 potatoes, 1 onion, 2 celery stalks, 4 cups broth, garlic, thyme, salt, pepper
Instructions: Dice all vegetables. Saute onion and garlic until soft. Add remaining vegetables and broth. Bring to boil. Simmer 30 minutes. Season with thyme, salt, and pepper.

Recipe: Pancakes
Ingredients: 1.5 cups flour, 3.5 tsp baking powder, 1 tbsp sugar, 1.25 cups milk, 1 egg, 3 tbsp butter
Instructions: Mix flour, baking powder, sugar. Make well in center. Pour in milk, egg, melted butter. Mix until smooth. Heat griddle. Pour batter. Cook until bubbles form. Flip and cook until golden.
"""

print(f"Recipe corpus: {len(RECIPE_CORPUS)} characters")
print(f"Preview:\n{RECIPE_CORPUS[:200]}...")


In [ ]:
from torch.utils.data import Dataset as TorchDataset

class TextDataset(TorchDataset):
    """Simple text dataset for fine-tuning: encode text with the tokenizer
    and return overlapping windows of token IDs."""
    def __init__(self, text, tokenizer, block_size=128):
        self.encodings = tokenizer.encode(text)
        self.block_size = block_size

    def __len__(self):
        return max(0, len(self.encodings) - self.block_size)

    def __getitem__(self, idx):
        chunk = self.encodings[idx : idx + self.block_size + 1]
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        return x, y


ft_dataset = TextDataset(RECIPE_CORPUS, pretrained_tokenizer, block_size=128)
ft_loader = DataLoader(ft_dataset, batch_size=4, shuffle=True, drop_last=True)
print(f"Fine-tuning dataset: {len(ft_dataset)} sequences of 128 tokens")
print(f"Batches per epoch: {len(ft_loader)}")


### Generate BEFORE fine-tuning

In [ ]:
# Generate recipe text BEFORE fine-tuning
prompt_recipe = "Recipe: Lemon"
torch.manual_seed(SEED)
result = generator(
    prompt_recipe,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7,
    num_return_sequences=1,
    pad_token_id=pretrained_tokenizer.eos_token_id,
)
print("BEFORE fine-tuning:")
print(result[0]["generated_text"])


### Fine-tuning loop

We do a manual training loop for transparency -- just 2 epochs is enough to see
the distribution shift. (For larger tasks you would use HuggingFace Trainer.)

In [ ]:
# Fine-tuning: manual loop for transparency
ft_model = GPT2LMHeadModel.from_pretrained("distilgpt2")
ft_model.train()

ft_optimizer = torch.optim.AdamW(ft_model.parameters(), lr=5e-5, weight_decay=0.01)

FT_EPOCHS = 2
ft_losses = []

print("Fine-tuning DistilGPT-2 on recipe corpus...")
print("=" * 60)

for epoch in range(1, FT_EPOCHS + 1):
    epoch_losses = []
    for xb, yb in ft_loader:
        outputs = ft_model(xb, labels=yb)
        loss = outputs.loss

        ft_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ft_model.parameters(), 1.0)
        ft_optimizer.step()

        epoch_losses.append(loss.item())
        ft_losses.append(loss.item())

    avg = np.mean(epoch_losses)
    print(f"Epoch {epoch}/{FT_EPOCHS} | Avg Loss: {avg:.4f}")

print("\nFine-tuning complete!")


In [ ]:
# Plot fine-tuning loss
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(ft_losses) + 1), ft_losses, marker="o", markersize=3, color="coral")
ax.set_xlabel("Training Step")
ax.set_ylabel("Cross-Entropy Loss")
ax.set_title("DistilGPT-2 Fine-Tuning Loss (Recipe Corpus)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Generate AFTER fine-tuning

In [ ]:
# Generate recipe text AFTER fine-tuning
ft_model.eval()
ft_generator = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=pretrained_tokenizer,
    device=-1,
)

torch.manual_seed(SEED)
result = ft_generator(
    "Recipe: Lemon",
    max_new_tokens=120,
    do_sample=True,
    temperature=0.7,
    num_return_sequences=1,
    pad_token_id=pretrained_tokenizer.eos_token_id,
)
print("AFTER fine-tuning:")
print(result[0]["generated_text"])


In [ ]:
# Try a few more prompts to demonstrate the shift
ft_prompts = [
    "Recipe: Chocolate",
    "Ingredients: 2 cups",
    "Instructions: Preheat",
]

print("=" * 70)
print("MORE FINE-TUNED GENERATIONS")
print("=" * 70)

for prompt in ft_prompts:
    torch.manual_seed(SEED)
    result = ft_generator(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        num_return_sequences=1,
        pad_token_id=pretrained_tokenizer.eos_token_id,
    )
    print(f"\nPrompt: {prompt!r}")
    print(result[0]["generated_text"])
    print("-" * 70)


### What Changed in the Model?

Fine-tuning did not add new "knowledge" to the model. What it did:

1. **Shifted the probability distribution** -- tokens like "Recipe:", "Ingredients:",
   "Preheat" became more likely in context.
2. **Adjusted attention patterns** -- the model now attends to recipe-structure cues.
3. **Preserved general English** -- the model still knows English grammar; it just
   preferentially generates recipe-like text now.

This is the same principle as Notebook 2's training dynamics: gradient descent moved
the weights to reduce loss on the new data, adjusting the loss landscape (curriculum
point 6).


---
# Part 5 -- Building a Gradio Demo

**Gradio** lets you build web UIs for ML models in a few lines of code.
This is the last step in the deployment stack: `weights -> inference -> API -> UI`.


In [ ]:
import gradio as gr

def generate_text(prompt, temperature, max_length, top_k):
    """Generate text using our fine-tuned model."""
    if not prompt.strip():
        return "Please enter a prompt."

    torch.manual_seed(42)
    result = ft_generator(
        prompt,
        max_new_tokens=int(max_length),
        do_sample=True,
        temperature=float(temperature),
        top_k=int(top_k),
        num_return_sequences=1,
        pad_token_id=pretrained_tokenizer.eos_token_id,
    )
    return result[0]["generated_text"]


# Build the Gradio interface
demo = gr.Interface(
    fn=generate_text,
    inputs=[
        gr.Textbox(label="Prompt", placeholder="Recipe: Chocolate...", lines=2),
        gr.Slider(minimum=0.1, maximum=2.0, value=0.7, step=0.1, label="Temperature"),
        gr.Slider(minimum=10, maximum=200, value=80, step=10, label="Max New Tokens"),
        gr.Slider(minimum=1, maximum=100, value=50, step=1, label="Top-k"),
    ],
    outputs=gr.Textbox(label="Generated Text", lines=8),
    title="Recipe Generator (Fine-tuned DistilGPT-2)",
    description="Generate cooking recipes using a fine-tuned language model. "
                "Adjust temperature (creativity), max tokens (length), and top-k (diversity).",
    examples=[
        ["Recipe: Lemon", 0.7, 100, 50],
        ["Ingredients: 1 cup flour", 0.8, 80, 50],
        ["Instructions: Preheat oven", 0.6, 80, 40],
    ],
)

# Display the interface inline (in Jupyter) or print a note
# NOTE: In Jupyter, demo.launch() opens a local server.
# We just show the interface definition here.
print("Gradio demo defined!")
print("To launch interactively, run: demo.launch()")
print()
print("The interface has:")
print("  - Text input for the prompt")
print("  - Temperature slider (0.1 to 2.0)")
print("  - Max tokens slider (10 to 200)")
print("  - Top-k slider (1 to 100)")
print("  - Generated text output")
print()
print("This is the deployment stack in action:")
print("  Model weights -> Inference function -> Gradio API -> Web UI")


### Deploying to Hugging Face Spaces

To make your demo permanently accessible:

1. Create a free account at [huggingface.co](https://huggingface.co)
2. Create a new Space (select Gradio as the SDK)
3. Upload your code and model files
4. The Space builds and deploys automatically -- you get a public URL

This completes the stack: **trained weights -> inference code -> API -> public web UI**.

The same architecture powers ChatGPT, Claude, and other AI products:
- A model (much larger) generates text
- An API wraps the model
- A web UI makes it accessible to users

The difference is scale (billions vs millions of parameters), safety (RLHF, filters),
and infrastructure (GPU clusters), but the *conceptual* stack is identical.


---
# Part 6 -- Reflection: What Deep Learning Is and Isn't

You have now completed the full journey from a single neuron (Notebook 1) to a
fine-tuned language model with a web interface. Let's step back and review
everything through the lens of our 7 curriculum points.


## The 7 Curriculum Points -- Revisited

### 1. Universal Approximation Theorem
*A single hidden layer can approximate any continuous function; depth = efficiency.*

- **Notebook 1**: You proved this by watching a tiny MLP learn a complex decision
  boundary on the moons dataset.
- **Here**: Our TinyGPT (2 layers) learned to mimic Shakespeare -- a very complex
  function from character sequences to probability distributions. More layers and
  attention heads = more efficient representation of these patterns.

### 2. ReLU Changed Everything
*Vanishing gradients with sigmoid/tanh; sparse activation; better gradient flow.*

- **Notebook 1**: You compared sigmoid vs ReLU gradient flow and saw sigmoid gradients
  vanish in deep networks.
- **Notebook 2**: Deep sigmoid networks failed to train; ReLU networks succeeded.
- **Here**: Modern transformers use GELU (a smooth cousin of ReLU) because it provides
  good gradient flow while being differentiable everywhere. The principle is the same:
  avoid saturating activations.

### 3. Overparameterization Helps
*More params than data, yet generalizes; implicit regularization of SGD.*

- **Notebook 2**: You trained an overparameterized MLP on a subset of MNIST and saw it
  generalize despite having more parameters than training samples.
- **Notebook 5**: The CNN experiments confirmed this pattern.
- **Here**: DistilGPT-2 has 82M parameters -- far more than the ~5KB of Shakespeare or
  the recipe corpus. Yet it generates coherent text. SGD + dropout + weight decay provide
  implicit regularization.

### 4. Neural Networks Don't "Understand" (This notebook's focus)
*They minimize loss, detect statistical structure, not meaning.*

- **Notebook 3**: Your character-level model generated plausible-looking text that was
  nonsensical -- pure statistical pattern matching.
- **Here**: DistilGPT-2 confidently answers "2+2 = " with wrong answers, invents facts
  about fictional countries, and cannot reverse strings. It has learned P(next_token | context),
  not reasoning, truth, or meaning.

### 5. Backpropagation = Chain Rule
*Efficient gradient computation via dynamic programming.*

- **Notebook 1**: You built the autograd engine and computed gradients by hand.
- **Notebook 2**: You manually derived and implemented backprop for matrix operations.
- **Here**: PyTorch's autograd computes gradients through the entire transformer
  (attention, layer norm, feed-forward, embeddings) automatically -- the same chain
  rule, just applied to a much deeper computation graph.

### 6. Geometry Matters
*High-dimensional loss landscape; flat minima -> better generalization.*

- **Notebook 2**: You visualized loss landscapes and saw how initialization affects
  which minimum SGD finds.
- **Notebook 5**: BatchNorm and residual connections smoothed the loss landscape.
- **Here**: LayerNorm, residual connections, and the AdamW optimizer all help
  navigate the loss landscape of a transformer. The learning rate warmup and cosine
  schedule we could add are also landscape navigation tools.

### 7. Capacity != Performance
*More layers != better; architecture, init, normalization, data quality dominate.*

- **Notebook 2**: You saw that a well-initialized small network beats a poorly
  initialized large one.
- **Notebook 5**: The ablation study showed architecture choices matter more than raw size.
- **Here**: Our 100K-parameter TinyGPT learned recognizable Shakespeare patterns.
  DistilGPT-2 (82M params) generates much better text, but even it makes mistakes.
  GPT-4 (rumored ~1T params) is much better still, but *still* does not understand.
  Adding parameters helps, but architecture (attention!), training data, and training
  recipe matter more.


## What the Model Learned vs. What It Didn't

| Learned | Did NOT Learn |
|---------|--------------|
| Statistical patterns in token sequences | Meaning, truth, or causation |
| Grammar and syntax (mostly) | Logic or reasoning |
| Domain-specific vocabulary (after fine-tuning) | Facts (it can only parrot training data patterns) |
| How to generate plausible continuations | Whether a statement is true |
| Attention patterns for long-range dependencies | Understanding of the physical world |

### The Deployment Stack

Throughout this series, we built up the full stack:

```
Raw math (Notebook 1: autograd, backprop)
    |
    v
Training recipes (Notebook 2: SGD, momentum, scheduling, init)
    |
    v
Generative modeling concepts (Notebook 3: P(x), temperature, sampling)
    |
    v
PyTorch framework (Notebook 4: tensors, autograd, nn.Module)
    |
    v
Computer vision (Notebook 5: CNNs, BatchNorm, augmentation)
    |
    v
Transformers + Deployment (Notebook 6: attention, fine-tuning, Gradio)
```

Each layer builds on the previous. You started by computing gradients by hand;
now you can fine-tune a pretrained transformer and deploy it with a web UI.


### What Comes Next

This series gave you the conceptual and practical foundations. From here you can:

1. **Scale up**: Use larger models (GPT-2, LLaMA) and GPU training
2. **Explore modalities**: Apply transformers to images (ViT), audio, video
3. **Learn RLHF**: How ChatGPT-style models are aligned with human preferences
4. **Study safety**: Adversarial attacks, jailbreaks, alignment research
5. **Build products**: Combine models with retrieval (RAG), tools, and agents

The conceptual toolkit you have -- autograd, loss landscapes, attention, transfer
learning, the gap between statistics and understanding -- will serve you in all
of these directions.


---
# Exercises

## Exercise 1: Implement Positional Encoding (Sinusoidal)

Our TinyGPT uses **learned** positional embeddings. The original "Attention Is All
You Need" paper used **sinusoidal** positional encodings instead:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

**Task:** Implement `SinusoidalPositionalEncoding` as an `nn.Module`. Then replace
the learned positional embedding in TinyGPT and retrain for 1 epoch. Compare the loss.


In [ ]:
# Exercise 1: Sinusoidal Positional Encoding

class SinusoidalPositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from Attention Is All You Need.

    Unlike learned positional embeddings, these are fixed (not trained).
    They have a nice property: the encoding for position p+k can be expressed
    as a linear function of the encoding for position p.
    """
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Register as buffer (not a parameter -- no gradients)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        """Add positional encoding to input embeddings.
        x: (batch, seq_len, d_model)
        """
        return x + self.pe[:, :x.size(1), :]


# Test it
spe = SinusoidalPositionalEncoding(d_model=64, max_len=256)
test_input = torch.zeros(1, 64, 64)  # batch=1, seq=64, dim=64
test_output = spe(test_input)

# Visualize the positional encodings
fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(spe.pe[0, :64, :].numpy(), aspect="auto", cmap="RdBu")
ax.set_xlabel("Embedding Dimension")
ax.set_ylabel("Position")
ax.set_title("Sinusoidal Positional Encodings (first 64 positions)")
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

print("Each row is the positional encoding for one position.")
print("Notice the sinusoidal patterns at different frequencies across dimensions.")
print("Lower dimensions oscillate slowly (capture coarse position);")
print("higher dimensions oscillate rapidly (capture fine position).")


## Exercise 2: Attention Pattern Analysis

After training the Tiny GPT, extract attention weights for a sample input and
visualize what each head has learned.

**Task:** Feed the string "ROMEO: To be or" through the trained model, extract
attention weights from each layer and head, and plot them as heatmaps.


In [ ]:
# Exercise 2 Solution: Attention Pattern Analysis

model.eval()

# Encode a sample and get attention weights from each block
sample_text = "ROMEO: To be or"
sample_ids = torch.tensor([[char_to_idx.get(c, 0) for c in sample_text]], dtype=torch.long)
sample_chars = list(sample_text)

# Forward pass, collecting attention weights
with torch.no_grad():
    tok = model.tok_emb(sample_ids)
    pos = model.pos_emb(torch.arange(sample_ids.size(1)))
    x = model.drop(tok + pos)
    mask = torch.tril(torch.ones(sample_ids.size(1), sample_ids.size(1))).unsqueeze(0).unsqueeze(0)

    all_weights = []
    for block in model.blocks:
        normed = block.ln1(x)
        _, weights_ex = block.attn(normed, mask=mask)
        all_weights.append(weights_ex)
        # Continue forward pass
        attn_out, _ = block.attn(normed, mask=mask)
        x = x + block.drop1(attn_out)
        normed = block.ln2(x)
        ffn_out = block.ffn(normed)
        x = x + block.drop2(ffn_out)

# Plot attention patterns
n_layers = len(all_weights)
n_heads = all_weights[0].size(1)
fig, axes = plt.subplots(n_layers, n_heads, figsize=(5 * n_heads, 4 * n_layers))
if n_layers == 1:
    axes = axes.reshape(1, -1)

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        ax = axes[layer_idx, head_idx]
        w = all_weights[layer_idx][0, head_idx].numpy()
        ax.imshow(w, cmap="Blues", vmin=0)
        ax.set_title(f"Layer {layer_idx+1}, Head {head_idx+1}")
        if head_idx == 0:
            ax.set_ylabel("Query")
        if layer_idx == n_layers - 1:
            ax.set_xlabel("Key")
        # Add character labels
        ax.set_xticks(range(len(sample_chars)))
        ax.set_xticklabels(sample_chars, rotation=90, fontsize=7)
        ax.set_yticks(range(len(sample_chars)))
        ax.set_yticklabels(sample_chars, fontsize=7)

plt.suptitle("Attention Patterns for: " + repr(sample_text), fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Different heads learn different attention patterns:")
print("  - Some heads attend to adjacent characters (local context)")
print("  - Some heads attend to specific characters like ':' or spaces (structural)")
print("  - The causal mask ensures all attention is lower-triangular")


## Exercise 3: Temperature vs. Perplexity Trade-off

Generate 200 characters at temperatures from 0.1 to 2.0. For each, compute the
perplexity of the generated text under the model.

**Perplexity** = $\exp(\text{average cross-entropy loss})$. Lower perplexity means
the model finds the text more "expected".

**Task:** Plot temperature vs. perplexity. What trend do you see?


In [ ]:
# Exercise 3 Solution: Temperature vs Perplexity

model.eval()

temperatures_sweep = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 1.3, 1.5, 2.0]
perplexities = []
gen_texts = []

prompt_ids_ex = torch.tensor([[char_to_idx["R"]]], dtype=torch.long, device=device)

for temp in temperatures_sweep:
    torch.manual_seed(SEED)
    gen_ids = model.generate(prompt_ids_ex, max_new_tokens=200, temperature=temp)
    gen_seq = gen_ids[0]

    # Compute perplexity: feed the generated text back through the model
    # and measure how "surprised" the model is
    with torch.no_grad():
        total_loss = 0.0
        count = 0
        for i in range(0, len(gen_seq) - 1, BLOCK_SIZE):
            chunk = gen_seq[i : i + BLOCK_SIZE + 1]
            if len(chunk) < 2:
                break
            x_chunk = chunk[:-1].unsqueeze(0)
            y_chunk = chunk[1:].unsqueeze(0)
            if x_chunk.size(1) > BLOCK_SIZE:
                x_chunk = x_chunk[:, :BLOCK_SIZE]
                y_chunk = y_chunk[:, :BLOCK_SIZE]
            _, loss_val = model(x_chunk, targets=y_chunk)
            total_loss += loss_val.item() * x_chunk.size(1)
            count += x_chunk.size(1)

        avg_loss = total_loss / max(count, 1)
        ppl = np.exp(avg_loss)
        perplexities.append(ppl)

    text = "".join(idx_to_char[i] for i in gen_seq.tolist())
    gen_texts.append(text[:60])

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(temperatures_sweep, perplexities, "o-", color="coral", linewidth=2, markersize=8)
ax.set_xlabel("Temperature", fontsize=12)
ax.set_ylabel("Perplexity", fontsize=12)
ax.set_title("Temperature vs. Perplexity of Generated Text", fontsize=14)
ax.grid(True, alpha=0.3)

# Annotate a few points
for i in [0, 4, -1]:
    ax.annotate(f"T={temperatures_sweep[i]}",
                xy=(temperatures_sweep[i], perplexities[i]),
                xytext=(10, 10), textcoords="offset points",
                fontsize=9, ha="left",
                arrowprops=dict(arrowstyle="->", color="gray"))

plt.tight_layout()
plt.show()

print("Observations:")
print("  - Low temperature: text is repetitive but has LOW perplexity")
print("    (the model is not surprised by repetitive output)")
print("  - High temperature: text is diverse but has HIGH perplexity")
print("    (random sampling produces unlikely sequences)")
print("  - Sweet spot around T=0.7-1.0: balanced quality and diversity")


---

## Congratulations!

You have completed the full deep learning curriculum. Starting from a single
neuron with manual gradient computation, you have arrived at fine-tuning
pretrained transformers and deploying them as web applications.

**The most important takeaway**: deep learning models are powerful pattern
matchers that learn statistical regularities from data. They do not understand,
reason, or know truth. The gap between "impressively fluent text" and "actual
understanding" is the central open question of AI research.

> "The question of whether machines can think is about as relevant as the
> question of whether submarines can swim." -- Edsger Dijkstra

What matters is not whether the model "understands," but whether we understand
what the model is doing -- and you now have the tools to investigate that.
